In [1]:
# =============================================================================
# Depth-sensitivity diagnostic: vertical structure by regime (upwelling vs relaxed)
# Run from the code/ dir (same as depth_profile_data.R). Raw depth-discrete data.
# =============================================================================
library(tidyverse)
source("depth_profile_data.R")   # also sources scenario_classification.R; loads C_TO_CHL_* consts

profile_data <- load_profile_data()

# Regime label per date (raw temp_50m classification). NOTE: boundary/transition
# months stay in 'relaxed' here — this is a general vertical-structure check, not
# the composite; refine later if a pattern hinges on those ~14 months.
regime_by_date <- profile_data$scenario %>%
  select(date, upwelling) %>%
  filter(!is.na(upwelling)) %>%
  distinct(date, .keep_all = TRUE)

# ---------------------------------------------------------------------------
# 1. NISKIN vertical structure by regime: NO3, Chl, PP, Temperature
# ---------------------------------------------------------------------------
niskin_reg <- profile_data$niskin %>%
  inner_join(regime_by_date, by = "date") %>%
  filter(depth <= 100) %>%
  mutate(depth_bin = round(depth / 5) * 5)

niskin_prof <- niskin_reg %>%
  group_by(upwelling, depth_bin) %>%
  summarize(
    n        = sum(!is.na(NO3_merged)),
    NO3_med  = median(NO3_merged, na.rm = TRUE),
    NO3_q25  = quantile(NO3_merged, .25, na.rm = TRUE),
    NO3_q75  = quantile(NO3_merged, .75, na.rm = TRUE),
    Chl_med  = median(Chlorophyll, na.rm = TRUE),
    PP_med   = median(PrimaryProductivity, na.rm = TRUE),
    Temp_med = median(Temperature, na.rm = TRUE),
    .groups  = "drop"
  ) %>% arrange(upwelling, depth_bin)

# Nitracline (first depth NO3 >= 1 uM) per regime
nitracline <- niskin_reg %>%
  filter(NO3_merged >= 1) %>%
  group_by(date, upwelling) %>% summarize(ncl = min(depth), .groups = "drop") %>%
  group_by(upwelling) %>%
  summarize(nitracline_med = median(ncl, na.rm = TRUE),
            q25 = quantile(ncl, .25, na.rm = TRUE),
            q75 = quantile(ncl, .75, na.rm = TRUE), .groups = "drop")

# ---------------------------------------------------------------------------
# 2. HPLC vertical structure by regime: size fractions per depth (Vidussi),
#    mirroring integrate_hplc() but POINTWISE (per depth, not integrated).
# ---------------------------------------------------------------------------
hplc_prof <- profile_data$hplc %>%
  inner_join(regime_by_date, by = "date") %>%
  filter(depth <= 100) %>%
  mutate(
    DP2 = 1.41*Fuco + 1.41*Perid + 0.60*Allo + 0.35*But_fuco +
          1.27*Hex_fuco + 0.86*Zea + 1.01*Tot_Chl_b,
    DP2 = ifelse(DP2 < 0.001, NA, DP2),
    micro = (1.41*Fuco + 1.41*Perid) / DP2,
    nano  = (0.60*Allo + 0.35*But_fuco + 1.27*Hex_fuco) / DP2,
    pico  = (0.86*Zea + 1.01*Tot_Chl_b) / DP2,
    micro_Cw = micro*C_TO_CHL_MICRO, nano_Cw = nano*C_TO_CHL_NANO, pico_Cw = pico*C_TO_CHL_PICO,
    tot_Cw = micro_Cw + nano_Cw + pico_Cw,
    micro_frac_N = micro_Cw/tot_Cw, nano_frac_N = nano_Cw/tot_Cw, pico_frac_N = pico_Cw/tot_Cw,
    size_centroid = micro_frac_N*log10(63) + nano_frac_N*log10(6.3) + pico_frac_N*log10(0.63),
    mean_cell_um  = 10^size_centroid,
    depth_bin = round(depth / 5) * 5
  ) %>%
  filter(!is.na(size_centroid)) %>%
  group_by(upwelling, depth_bin) %>%
  summarize(
    n             = n(),
    micro_frac    = median(micro_frac_N, na.rm = TRUE),
    nano_frac     = median(nano_frac_N,  na.rm = TRUE),
    pico_frac     = median(pico_frac_N,  na.rm = TRUE),
    mean_cell_um  = median(mean_cell_um, na.rm = TRUE),
    .groups = "drop"
  ) %>% arrange(upwelling, depth_bin)

cat("===== Nitracline depth (first NO3>=1) by regime =====\n"); print(nitracline)
cat("\n===== NISKIN profile by regime (5 m bins) =====\n"); print(niskin_prof, n = 200)
cat("\n===== HPLC size structure by regime (5 m bins) =====\n"); print(hplc_prof, n = 200)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Lade nötiges Paket: gsw



Loaded HPLC profiles: 176 dates, depths 0-200m
Interpolating Niskin data (this may take a moment)...
  Interpolating NO3_merged...
  Interpolating Chlorophyll...
  Interpolating Phaeopigments...
  Interpolating PrimaryProductivity...
  Interpolating PN_ug_L...
  Interpolating Temperature...
Saved Niskin profiles to cache: ../processed/Niskin_depth_profiles.rds
  Dates: 230, depths 0-200m

=== Date Coverage Summary ===
  Total unique dates:     275
  With HPLC data:         176
  With Niskin data:       230
  With scenario class:    257
  With observed EuZ:      102
===== Nitracline depth (first NO3>=1) by regime =====
# A tibble: 2 × 4
  upwelling nitracline_med   q25   q75
  <chr>              <dbl> <dbl> <dbl>
1 relaxed               40    29  48  
2 upwelling             18     8  26.2

===== NISKIN profile by regime (5 m bins) =====
# A tibble: 42 × 9
   upwelling depth_bin     n NO3_med NO3_q25 NO3_q75 Chl_med  PP_med Temp_med
   <chr>         <dbl> <int>   <dbl>   <dbl>   <dbl>  

In [2]:
# =============================================================================
# Validate the dynamic per-month EuZ choice: per-month EuZ distribution by regime
# + regime-composite robustness across the few real depth-mode choices.
# Run from code/ ; reuses profile_data / proxy_model if already in the session.
# =============================================================================
library(tidyverse)
source("depth_profile_data.R")

if (!exists("profile_data")) profile_data <- load_profile_data()
if (!exists("proxy_model"))
  proxy_model <- fit_euphotic_proxy(profile_data$scenario,
                                    niskin_profiles = profile_data$niskin,
                                    model_type = "isotherm_chl")

# Regime composite on HPLC months (phyto mmolN present), median — matches the
# locked-in cariaco_obs choice (months='hplc', agg='median').
regime_stats <- function(df) {
  phyto <- c("micro_mmolN", "nano_mmolN", "pico_mmolN")
  df %>%
    filter(regime_adjusted %in% c("upwelling", "relaxed"),
           if_all(all_of(phyto), ~ !is.na(.))) %>%
    mutate(mean_cell_um = 10^size_centroid,
           FN_per_vol   = FN_mmolN_m2_d / depth_cutoff) %>%
    group_by(regime_adjusted) %>%
    summarize(n = n(),
              depth      = median(depth_cutoff, na.rm = TRUE),
              mean_cell  = median(mean_cell_um, na.rm = TRUE),
              micro      = median(micro_frac_N, na.rm = TRUE),
              nano       = median(nano_frac_N,  na.rm = TRUE),
              pico       = median(pico_frac_N,  na.rm = TRUE),
              NO3        = median(NO3_mmolN, na.rm = TRUE),
              PP         = median(PP_mgC_m2_d, na.rm = TRUE),
              FN         = median(FN_mmolN_m2_d, na.rm = TRUE),
              FN_per_vol = median(FN_per_vol, na.rm = TRUE),
              .groups = "drop")
}

modes <- list(dynamic  = list(depth_mode = "dynamic"),
              observed = list(depth_mode = "observed"),
              scenario = list(depth_mode = "scenario"),
              fixed50  = list(depth_mode = "fixed", fixed_depth = 50))

fds <- map(modes, function(a)
  get_full_scenario_data(profile_data,
    depth_mode      = a$depth_mode,
    fixed_depth     = if (is.null(a$fixed_depth)) 50 else a$fixed_depth,
    scenario_depths = c(upwelling = 35, relaxed = 50),
    proxy_model     = proxy_model))

cat("\n===== Regime composite across depth modes (HPLC months, median) =====\n")
print(imap_dfr(fds, ~ regime_stats(.x) %>% mutate(mode = .y, .before = 1)) %>%
        arrange(regime_adjusted, mode), n = 100, width = Inf)

# Per-month EuZ distribution under dynamic, split by observed vs proxy-predicted
cat("\n===== Dynamic per-month EuZ by regime and source =====\n")
fds$dynamic %>%
  filter(regime_adjusted %in% c("upwelling", "relaxed"), !is.na(depth_cutoff)) %>%
  group_by(regime_adjusted, cutoff_source) %>%
  summarize(n = n(),
            euz_med = median(depth_cutoff), q25 = quantile(depth_cutoff, .25),
            q75 = quantile(depth_cutoff, .75),
            euz_min = min(depth_cutoff), euz_max = max(depth_cutoff),
            .groups = "drop") %>%
  print(width = Inf)

Chlorophyll integration (0-100m):
  Dates with sufficient Chl data: 229

Fitting EuZ proxy model (isotherm_chl) on 94 observations...
  Model: EuZ = 81.55 + 0.0983 × Isotherm_21 + -30.4234 × log10(Chl_int)
  Coefficient signs: Isotherm ✓ (expect +), log(Chl) ✓ (expect -)
  R-squared: 0.717 (adj: 0.710)
  RMSE: 6.94 m

Prediction coverage:
  Total dates with Isotherm_21: 229
  With observed EuZ: 94
  With predicted EuZ (iso+chl): 104
  No prediction (missing Chl): 31

=== Building full scenario dataset (mode: dynamic) ===
Complete monthly backbone: 255 months
  From 1995-11-01 to 2017-01-01
  With upwelling class: 228 months
  Boundary reclassification: 14 relaxed months -> 'transition' (buffer=0.50 C, window=1)
  Dynamic cutoffs from isotherm_chl model
  With depth cutoff: 198 months
HPLC: 153 months with data
Niskin: 198 months with data
F_N computed for 186 months
Zooplankton: 153 months with data
Sediment trap: 160 months with data (lag = 0)

=== Final Data Coverage ===
  Total mont

In [6]:
# =============================================================================
# Test: surface / upper-layer Chl as EuZ predictor vs 0-100 m integrated Chl.
# =============================================================================
library(tidyverse)
source("depth_profile_data.R")
if (!exists("profile_data")) profile_data <- load_profile_data()

# Fit set: months with observed EuZ + isotherm (already carries `upwelling`)
scen <- profile_data$scenario %>%
  filter(!is.na(euphotic_depth_obs), !is.na(Isotherm_21)) %>%
  distinct(date, .keep_all = TRUE)

chl_mean_to <- function(upper) {
  profile_data$niskin %>%
    filter(depth <= upper, !is.na(Chlorophyll)) %>%
    group_by(date) %>%
    summarize(chl = mean(Chlorophyll, na.rm = TRUE), nd = n(), .groups = "drop") %>%
    filter(nd >= max(3, upper / 2)) %>%
    transmute(date, log_chl = log10(chl + 0.1))
}

chl_int100 <- profile_data$niskin %>%
  filter(depth <= 100, !is.na(Chlorophyll)) %>%
  group_by(date) %>%
  summarize(chl = sum(Chlorophyll, na.rm = TRUE), nd = n(), .groups = "drop") %>%
  filter(nd >= 50) %>% transmute(date, log_chl = log10(chl + 0.1))

candidates <- list("integrated_0-100m (current)" = chl_int100,
                   "surface_0-10m" = chl_mean_to(10),
                   "upper_0-20m"   = chl_mean_to(20),
                   "upper_0-30m"   = chl_mean_to(30))

fit_and_diag <- function(name, chl_df) {
  d <- scen %>% inner_join(chl_df, by = "date")          # d keeps scen$upwelling
  m <- lm(euphotic_depth_obs ~ Isotherm_21 + log_chl, data = d)
  d$pred  <- predict(m)
  d$resid <- d$pred - d$euphotic_depth_obs               # +ve = over-predicts (too deep)
  reg <- d %>% filter(!is.na(upwelling)) %>% group_by(upwelling) %>%
    summarize(bias = mean(resid), obs = median(euphotic_depth_obs),
              pred = median(pred), .groups = "drop")
  g <- function(r, c) reg[[c]][reg$upwelling == r]
  tibble(model = name, n = nrow(d),
         R2 = summary(m)$r.squared, RMSE = sqrt(mean(d$resid^2)),
         up_bias = g("upwelling", "bias"), rel_bias = g("relaxed", "bias"),
         up_obs = g("upwelling", "obs"), up_pred = g("upwelling", "pred"),
         rel_obs = g("relaxed", "obs"), rel_pred = g("relaxed", "pred"))
}

print(imap_dfr(candidates, ~ fit_and_diag(.y, .x)), width = Inf)

# A tibble: 4 × 10
  model                           n    R2  RMSE up_bias rel_bias up_obs up_pred
  <chr>                       <int> <dbl> <dbl>   <dbl>    <dbl>  <dbl>   <dbl>
1 integrated_0-100m (current)    94 0.717  6.94   1.25    -0.845   35.3    38.9
2 surface_0-10m                  94 0.745  6.59   0.791   -0.537   35.3    37.0
3 upper_0-20m                    94 0.777  6.16   0.682   -0.463   35.3    38.9
4 upper_0-30m                    94 0.791  5.96   0.749   -0.508   35.3    38.5
  rel_obs rel_pred
    <dbl>    <dbl>
1    52.5     51.3
2    52.5     51.9
3    52.5     52.3
4    52.5     52.8
